In [1]:
import pandas as pd
from pathlib import Path

DATA = Path("data")
test_df = pd.read_parquet(DATA / "test_clean.parquet")
preds = pd.read_parquet(DATA / "test_preds.parquet").set_index("SK_ID_CURR")
test_df["p"] = preds.loc[test_df["SK_ID_CURR"], "p_lgb"].values

bad  = test_df["TARGET"] == 1
good = ~bad
bad_money = test_df.loc[bad, "AMT_CREDIT"].sum()

rows = []
for decline_pct in [5, 10, 15, 20, 30, 40]:
    cut = test_df["p"].quantile(1 - decline_pct / 100)
    dec = test_df["p"] >= cut
    rows.append({
        "decline_%": decline_pct,
        "cutoff_risk_%": round(cut * 100, 1),
        "defaulters_caught_%": round((dec & bad).sum() / bad.sum() * 100, 1),
        "good_customers_lost_%": round((dec & good).sum() / good.sum() * 100, 1),
        "money_at_risk_avoided_%": round(test_df.loc[dec & bad, "AMT_CREDIT"].sum() / bad_money * 100, 1),
        "approved_default_%": round(test_df.loc[~dec, "TARGET"].mean() * 100, 2),
    })

tradeoff = pd.DataFrame(rows)
print("Default rate if we approve everyone:", round(test_df["TARGET"].mean() * 100, 2), "%\n")
tradeoff

Default rate if we approve everyone: 8.07 %



,decline_%,cutoff_risk_%,defaulters_caught_%,good_customers_lost_%,money_at_risk_avoided_%,approved_default_%
0,5,24.9,21.8,3.5,19.6,6.65
1,10,17.9,35.3,7.8,32.6,5.80
2,15,14.0,45.4,12.3,42.2,5.19
3,20,11.6,54.5,17.0,51.0,4.59
4,30,8.4,67.2,26.7,63.9,3.78
5,40,6.4,76.3,36.8,73.3,3.19


In [2]:
import numpy as np

test_df["band"] = np.select([test_df["p"] >= 0.25, test_df["p"] >= 0.12],
                            ["decline", "review"], default="approve")

summary = test_df.groupby("band").agg(people=("TARGET", "size"),
                                      default_pct=("TARGET", "mean"))
summary["share_%"] = (summary["people"] / len(test_df) * 100).round(1)
summary["default_pct"] = (summary["default_pct"] * 100).round(2)
print(summary.loc[["approve", "review", "decline"]], "\n")

test_df["age_group"] = pd.cut(test_df["age_years"], [20, 30, 40, 50, 60, 70],
                              labels=["20-29", "30-39", "40-49", "50-59", "60-69"])
gp = test_df[test_df["TARGET"] == 0]
print("Good payers: % auto-declined vs sent to review, by age")
print((pd.crosstab(gp["age_group"], gp["band"], normalize="index") * 100).round(2)
      [["approve", "review", "decline"]])

         people  default_pct  share_%
band                                 
approve   49875         4.69     81.1
review     8578        18.13     13.9
decline    3050        35.18      5.0 

Good payers: % auto-declined vs sent to review, by age
band       approve  review  decline
age_group                          
20-29        69.84   22.69     7.47
30-39        79.54   15.51     4.95
40-49        85.84   11.26     2.91
50-59        90.19    8.21     1.61
60-69        95.17    4.26     0.57


## Notebook 08: choosing a cut-off

There is no perfect cut-off. Declining more people stops more bad loans but also turns away good customers. For example, declining the riskiest 10% catches 35% of defaulters while losing 8% of good customers.

I used three bands instead of yes/no:

| Band | Rule | Share | Default rate |
|---|---|---|---|
| Approve | risk under 12% | 81.1% | 4.69% |
| Human review | 12% to 25% | 13.9% | 18.13% |
| Decline | 25% or more | 5.0% | 35.18% |

The default rate among approved loans drops from 8.07% to 4.69%.

Fairness: good payers aged 20 to 29 now get auto-declined 7.5% of the time, down from 31.6% under a simple top-20% rule. Most go to a human instead. The age gap still exists in relative terms, so reviewers should keep watching it.

Limitation: I picked 12% and 25% by looking at test-set results. A real rollout should set them on separate data.